# Tuning lab — move one knob, watch the score

Pick your cluster, then change **one** lever at a time. Every run re-scores
**t-SNE / UMAP / EVoC** against the kinematic ground truth (Gaia parallax +
proper motion + radial velocity). Your goal: beat the baseline in
`docs/region_sweep_results.md` — and be able to say *which* knob moved the
number and *why*.

The three frontier tracks live in `docs/student_activities.md`
(spectral latent · isochrone+red-clump · GALAH cross-match).

**Iterating fast:** changing a control re-runs the render callback below in
place. Repeated configurations come back from the notebook's own memo, and the
prepared sample is cached on disk (`prepare` ≈ 0.04 s after the first ~20 s
read). A genuinely new configuration means a real computation, as it should.

### Running it

Inside the workshop container, like `chemical_tagging.ipynb`:

```bash
export IMG=ghcr.io/iaa-so-training/day4-clustering:latest
export DAY4="-v $PWD/data:/app/data -v $PWD/results:/app/results -v $PWD/notebooks:/app/notebooks"

docker run --rm -it $DAY4 $IMG uv run cluster download --all      # catalogue + embeddings, once
docker run --rm -it -p 8889:8889 $DAY4 $IMG \
  uv run jupyter lab --ip=0.0.0.0 --port=8889 --no-browser --IdentityProvider.token=""
```

Then open this notebook (`notebooks/tuning_template.ipynb`) at
http://localhost:8889 and run it top to bottom —
**Kernel → Restart Kernel and Run All Cells**.

In [ ]:
import os
from pathlib import Path

_PROJECT_ROOT = Path.cwd()
if not (_PROJECT_ROOT / "data").is_dir() and (_PROJECT_ROOT.parent / "data").is_dir():
    _PROJECT_ROOT = _PROJECT_ROOT.parent
os.chdir(_PROJECT_ROOT)

import ipywidgets as widgets
import pandas as pd
from IPython.display import Markdown, clear_output, display

from cluster import config
from cluster.benchmark import run_benchmark as _run_benchmark
from cluster.clusters import CLUSTERS
from cluster.data import prepare as _prepare

allstar = Path("data/astraAllStarASPCAP-0.6.0.fits.gz")
if not allstar.exists():
    raise FileNotFoundError(
        "data/astraAllStarASPCAP-0.6.0.fits.gz not found — from your checkout run "
        "`docker run --rm -it $DAY4 $IMG uv run cluster download --all`, "
        "or `uv run cluster download --all` inside the container."
    )

In [ ]:
cluster_dropdown = widgets.Dropdown(
    options=sorted(c.name for c in CLUSTERS), value="M 67", description="cluster",
)
region = widgets.BoundedFloatText(value=30.0, min=1.0, max=60.0, step=1.0, description="region (°)")
use_weights = widgets.Checkbox(value=False, description="element 1/σ weights")
normalize = widgets.Checkbox(value=True, description="row-normalise (L2)")
perplexity = widgets.BoundedIntText(value=30, min=5, max=100, step=5, description="perplexity")
min_cluster = widgets.BoundedIntText(value=5, min=2, max=50, step=1, description="min_cluster_size")

widgets.VBox([
    cluster_dropdown,
    widgets.HBox([region, perplexity, min_cluster]),
    widgets.HBox([use_weights, normalize]),
])

In [ ]:
# One render callback, re-run whenever a control changes. Results are memoised on
# the full configuration, so re-picking a setting you tried before is instant.
_MEMO: dict[str, object] = {}


def _render(*_change):
    settings = config.Settings()
    settings.cluster_names = [cluster_dropdown.value]
    settings.region_radius_deg = region.value
    settings.use_element_weights = use_weights.value
    settings.normalize_rows = normalize.value
    settings.tsne["perplexity"] = perplexity.value
    settings.hdbscan["min_cluster_size"] = min_cluster.value

    clusters = [c for c in CLUSTERS if c.name == cluster_dropdown.value]
    key = settings.model_dump_json() + "|" + clusters[0].name
    if key not in _MEMO:
        prepared = _prepare(
            allstar, settings, clusters,
            seed_position_radius_deg=config.SEED_POSITION_RADIUS_DEG,
            seed_parallax_frac=config.SEED_PARALLAX_FRAC,
            seed_pm_tol=config.SEED_PM_TOL,
            seed_rv_tol=config.SEED_RV_TOL,
            n_refine_passes=config.N_REFINE_PASSES,
            refine_sigma=config.REFINE_SIGMA,
        )
        _MEMO[key] = (prepared, _run_benchmark(prepared, settings))
    prepared, result = _MEMO[key]

    frames = []
    for name, r in result.results.items():
        f = r.scores.copy()
        f.insert(0, "method", name)
        frames.append(f)
    scores = pd.concat(frames, ignore_index=True)

    clear_output(wait=True)   # redraw this cell in place
    display(Markdown(
        f"### {cluster_dropdown.value} — region {region.value}° · "
        f"weights={use_weights.value} · normalize={normalize.value} · "
        f"perplexity={perplexity.value} · min_cluster={min_cluster.value}"
    ))
    display(scores)


for _w in (cluster_dropdown, region, use_weights, normalize, perplexity, min_cluster):
    _w.observe(_render, names="value")

_render()

## How to read it

- **recall** — fraction of the cluster's true members the best-overlapping
  predicted cluster recovers.
- **precision** — purity of that predicted cluster (field contamination pulls it
  down).
- A recall jump with a precision collapse means your change swallowed the field
  into one blob — that's a lesson, not a win.

Log what you tried; your deliverable is *one* change and why it moved the
numbers.